# SQS суперячейки 128 атомов: Al-Co-Cr-Fe-Ni-Ti (L12)
Упрочняющая фаза **(Ni,Co,Fe,Cr)₃(Al,Ti)** со структурой **L12** (Pm-3m, #221)

Генерация **Special Quasirandom Structure (SQS)** с помощью **sqsgenerator**.
- Размер: **128 атомов** (4×4×2 суперячейка)
- Режим: `sublattice_mode: split` (строгое разделение A/B подрешёток)
- Итерации: **50 000 000** (высокая точность оптимизации SRO)

# Импорт

In [1]:
%pip install sqsgenerator pymatgen

   ---------------------------------------- 0.0/981.0 kB ? eta -:--:--
   ---------- ----------------------------- 262.1/981.0 kB ? eta -:--:--
   ---------------------------------------- 981.0/981.0 kB 5.9 MB/s  0:00:00

   ---------------------------------------- 0/2 [click]
   ---------------------------------------- 0/2 [click]
   ---------------------------------------- 0/2 [click]
   -------------------- ------------------- 1/2 [sqsgenerator]
   -------------------- ------------------- 1/2 [sqsgenerator]
   -------------------- ------------------- 1/2 [sqsgenerator]
   ---------------------------------------- 2/2 [sqsgenerator]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from sqsgenerator import parse_config, optimize, write
from pymatgen.core import Structure

# Конфигурация sqsgenerator

In [2]:
a = 3.57
nx, ny, nz = 4, 4, 2  # 4x4x2 -> 32 ячейки * 4 атома = 128 атомов

In [3]:
coords = []
species = []

for ix in range(nx):
    for iy in range(ny):
        for iz in range(nz):
            # B-сайт (1a)
            coords.append([ix/nx, iy/ny, iz/nz])
            species.append("Al")
            # A-сайты (3c)
            for offset in [[0.5, 0.5, 0], [0.5, 0, 0.5], [0, 0.5, 0.5]]:
                coords.append([(ix+offset[0])/nx, (iy+offset[1])/ny, (iz+offset[2])/nz])
                species.append("Ni")

config = {
    "iterations": 50000000,
    "sublattice_mode": "split",
    "structure": {
        "lattice": [[a*nx, 0, 0], [0, a*ny, 0], [0, 0, a*nz]],
        "coords": coords,
        "species": species,
        "supercell": [1, 1, 1]
    },
    "composition": [
        # B-подрешётка (32 сайта): Al:Ti = 3:1 -> 24 Al, 8 Ti
        {"sites": "Al", "Al": 24, "Ti": 8},
        # A-подрешётка (96 сайтов): Ni:Co:Fe:Cr = 12:5:4:3 -> 48 Ni, 20 Co, 16 Fe, 12 Cr
        {"sites": "Ni", "Ni": 48, "Co": 20, "Fe": 16, "Cr": 12}
    ],
    "max_results_per_objective": 5
}

print(f"Сгенерировано сайтов: {len(coords)}")

Сгенерировано сайтов: 128


## 3. Запуск оптимизации SQS

In [4]:
cfg = parse_config(config)
pack = optimize(cfg)
print(f"Найдено решений: {len(pack)}")
for i, (obj, sols) in enumerate(pack):
    print(f"Objective #{i}: {obj:.6f} ({len(sols)} структур)")

Найдено решений: 35
Objective #0: 9.130166 (1 структур)
Objective #1: 9.145708 (1 структур)
Objective #2: 9.173015 (1 структур)
Objective #3: 9.180345 (1 структур)
Objective #4: 9.290550 (1 структур)
Objective #5: 9.317115 (1 структур)
Objective #6: 9.327134 (1 структур)
Objective #7: 9.680072 (1 структур)
Objective #8: 9.756309 (1 структур)
Objective #9: 9.756896 (1 структур)
Objective #10: 9.757150 (1 структур)
Objective #11: 9.804728 (1 структур)
Objective #12: 9.838077 (1 структур)
Objective #13: 10.031338 (1 структур)
Objective #14: 10.123649 (1 структур)
Objective #15: 10.184818 (1 структур)
Objective #16: 10.441806 (1 структур)
Objective #17: 10.587364 (1 структур)
Objective #18: 10.830572 (1 структур)
Objective #19: 10.952078 (1 структур)
Objective #20: 11.156506 (1 структур)
Objective #21: 11.443611 (1 структур)
Objective #22: 11.772810 (1 структур)
Objective #23: 11.912078 (1 структур)
Objective #24: 11.949151 (1 структур)
Objective #25: 12.158502 (1 структур)
Objective #26: 

## Экспорт лучшего SQS

In [5]:
best = pack.best()
print(best.objective)

9.130165598290596


In [6]:
write(best.structure(), "HEA_L12_SQS_128atoms.cif")

In [7]:
!sed -i 's/0+//g' HEA_L12_SQS_128atoms.cif

"sed" �� ���� ����७��� ��� ���譥�
��������, �ᯮ��塞�� �ணࠬ��� ��� ������ 䠩���.


## 5. Проверка структуры

In [8]:
sqs_128 = Structure.from_file("HEA_L12_SQS_128atoms.cif")

print(f"Формула: {sqs_128.composition.formula}")
print(f"Число атомов: {sqs_128.num_sites}")
print(f"Решётка: {sqs_128.lattice.a:.2f} x {sqs_128.lattice.b:.2f} x {sqs_128.lattice.c:.2f}")

for el, amt in sqs_128.composition.as_dict().items():
    print(f"{el}: {amt:.0f} ({amt/sqs_128.num_sites*100:.1f}%)")

Формула: Ti8 Al24 Cr12 Fe16 Co20 Ni48
Число атомов: 128
Решётка: 14.28 x 14.28 x 7.14
Ti0+: 8 (6.2%)
Al0+: 24 (18.8%)
Cr0+: 12 (9.4%)
Fe0+: 16 (12.5%)
Co0+: 20 (15.6%)
Ni0+: 48 (37.5%)
